# Diseño inicial de `analytics.fact_orders`

## Grano de la tabla

La tabla `analytics.fact_orders` tendrá una granularidad de **una fila por pedido**.

**1 fila = 1 `order_id`**

Esto significa que cualquier información procedente de tablas con varias filas por pedido debe agregarse previamente a nivel de `order_id`.

La estructura conceptual de la tabla será:

`fact_orders = orders + items_by_order + payments_by_order + reviews_by_order`

## Columnas propuestas para `analytics.fact_orders`

| Columna en `fact_orders` | Procedencia | Tipo | ¿Agregación? | Explicación |
|---|---|---|---|---|
| `order_id` | `core.orders.order_id` | Identificador | No | Identificador único del pedido. Será la clave principal de la tabla de hechos. |
| `customer_id` | `core.orders.customer_id` | Identificador / FK | No | Cliente asociado al pedido. Puede funcionar después como clave hacia `dim_customer`. |
| `order_status` | `core.orders.order_status` | Categórica | No | Estado del pedido. Puede convertirse después en dimensión. |
| `order_purchase_timestamp` | `core.orders.order_purchase_timestamp` | Fecha | No | Fecha y hora de compra del pedido. |
| `order_approved_at` | `core.orders.order_approved_at` | Fecha | No | Fecha y hora de aprobación del pedido. Puede contener nulos. |
| `order_delivered_carrier_date` | `core.orders.order_delivered_carrier_date` | Fecha | No | Fecha en la que el pedido fue entregado al transportista. Puede contener nulos. |
| `order_delivered_customer_date` | `core.orders.order_delivered_customer_date` | Fecha | No | Fecha real de entrega al cliente. Puede contener nulos. |
| `order_estimated_delivery_date` | `core.orders.order_estimated_delivery_date` | Fecha | No | Fecha estimada de entrega. |
| `approval_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo entre compra y aprobación del pedido. |
| `carrier_dispatch_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo entre aprobación y entrega al transportista. |
| `delivery_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo entre entrega al transportista y entrega al cliente. |
| `total_delivery_time_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Tiempo total desde compra hasta entrega al cliente. |
| `delay_days` | Derivada de `core.orders` | Métrica temporal | Derivada | Diferencia entre entrega real y entrega estimada. |
| `is_delayed` | Derivada de `delay_days` | Binaria | Derivada | 1 si el pedido llegó tarde, 0 si llegó a tiempo o antes. |
| `total_items_value` | `core.order_items.price` | Métrica económica | Sí | Suma del precio de todos los ítems del pedido. |
| `total_freight_value` | `core.order_items.freight_value` | Métrica económica/logística | Sí | Suma del coste de envío de todas las líneas del pedido. |
| `number_of_items` | `core.order_items.order_item_id` | Métrica de conteo | Sí | Número total de líneas de pedido asociadas al pedido. |
| `number_of_products` | `core.order_items.product_id` | Métrica de conteo | Sí | Número de productos distintos dentro del pedido. |
| `number_of_sellers` | `core.order_items.seller_id` | Métrica de conteo | Sí | Número de vendedores distintos involucrados en el pedido. |
| `total_payment_value` | `core.order_payments.payment_value` | Métrica económica | Sí | Suma del valor pagado en todas las operaciones de pago del pedido. |
| `number_of_payment_operations` | `core.order_payments.payment_sequential` | Métrica de conteo | Sí | Número de registros de pago asociados al pedido. |
| `max_payment_installments` | `core.order_payments.payment_installments` | Métrica financiera | Sí | Máximo número de cuotas usado en alguna operación de pago del pedido. |
| `avg_review_score` | `core.order_reviews.review_score` | Métrica de satisfacción | Sí | Puntuación media de las reseñas asociadas al pedido. |
| `number_of_reviews` | `core.order_reviews.review_id` | Métrica de conteo | Sí | Número de reseñas asociadas al pedido. |